# Atividade Prática – CI_Parte 2 - Aula 5 – Integração DevOps
**Curso:** Ciência da Computação  

**Professor:** Danilo Silva  

**Tema:** Construindo e Executando Pipelines de CI na Prática via Google Colab

**Objetivo:** Criar um script de pipeline (`workflow` em YAML) do zero, compreender a hierarquia de *Workflows / Jobs / Steps*, automatizar o build/testes, simular a execução e realizar o **deploy automatizado** do artefato na pasta compartilhada do Google Drive do professor.

## ETAPA 1 E 2: PREPARAÇÃO DO AMBIENTE NO GOOGLE COLAB (EXECUTAR NO TERMINAL)

Nesta etapa, iremos clonar o projeto base da Aula 4, preparar a estrutura de arquivos no Colab e instalar a biblioteca `act` (ferramenta para rodar GitHub Actions localmente) no ambiente Linux do Colab.

## ETAPA 3: CRIAÇÃO DO WORKFLOW YAML DO ZERO

Nesta etapa, você criará a estrutura do arquivo do pipeline declarando explicitamente a hierarquia: **Workflow > Job > Steps**.

In [ ]:
import os

%cd devops_aula4ceub

# Cria a estrutura de pastas .github/workflows
#!mkdir -p .github/workflows

# Conteúdo do pipeline de CI em formato YAML
workflow_content = """
name: Pipeline de Integracao Continua (CI)

on:
  push:
    branches: [ "main" ]
  pull_request:
    branches: [ "main" ]

jobs:
  build-and-test:
    runs-on: ubuntu-latest

    steps:
      # Step 1: Clona o repositório no runner
      - name: Checkout do Repositorio
        uses: actions/checkout@v4

      # Step 2: Instalação de dependências do Node.js
      - name: Instalar Dependencias
        run: npm ci || npm install

      # Step 3: Análise Estática de Código (Linter)
      - name: Executar Linter (ESLint)
        run: npm run lint

      # Step 4: Suíte de Testes Automatizados (Jest)
      - name: Executar Testes
        run: npm test

      # Step 5: Processo de Build do projeto
      - name: Executar Build
        run: npm run build --if-present

      # Step 6: Exportação dos artefatos de Build no ambiente local
      - name: Exportar Artefato de Build
        run: |
          mkdir -p ./dist_artefatos
          cp -r src/ ./dist_artefatos/
          echo "Build finalizado e entregue em $(date)" > ./dist_artefatos/build_info.txt
"""

# Escreve o conteúdo no arquivo .github/workflows/pipeline_ci.yml
workflow_path = ".github/workflows/pipeline_ci.yml"
with open(workflow_path, "w") as f:
    f.write(workflow_content)

print(f"Arquivo '{workflow_path}' criado com sucesso.")


## ETAPA 4: INSTALAÇÃO DO NPM E EXECUÇÃO DE TESTES

Executaremos os comandos da etapa 4 presentes no arquivo da atividade.

## ETAPA 5: EXECUÇÃO DO PIPELINE NO AMBIENTE COLAB

Executaremos o ciclo completo da esteira de CI no ambiente do Colab, testando a instalação de dependências, linter, testes e a geração do build.

In [ ]:
# Entrega oficial do artefato gerado no sistema de arquivos do Colab

!mkdir -p ./build_output
!cp -r src/ ./build_output/
!echo "Build gerada com sucesso via pipeline em $(date)" > ./build_output/status_build.txt

print("\n--- Conteúdo da Build Gerada ---")
!ls -la ./build_output

## ETAPA 6: EXPERIMENTANDO AS BARREIRAS DA ESTEIRA DE CI

### Teste A: Simulando Falha no Linter (Análise Estática)

In [ ]:
# Força um erro no Linter adicionando uma variável não utilizada no código
!echo "const variavelSemUso = 999;" >> src/calculadora.js

# Executa o Linter e observa a falha interrompendo o fluxo
!npm run lint

### Teste B: Corrigindo o Linter e Induzindo Erro no Teste Automatizado

In [ ]:
# Restaura o código sem erro de sintaxe, mas altera a regra de negócio (somar fazendo subtração)
code_bug = '''\
function somar(a, b) {
  return a - b; // Erro proposital de lógica
}
module.exports = { somar };
'''
with open('src/calculadora.js', 'w') as f:
    f.write(code_bug)

# Executa Linter (passa) e Testes (falha)
!npm run lint && npm test

## ETAPA 7: RESTAURAÇÃO, BUILD FINAL E COMPACTAÇÃO DOS ARTEFATOS

Nesta etapa final, restauramos a regra de negócio correta, executamos o pipeline completo e empacotamos o artefato final (`.tar.gz`) no ambiente do Google Colab para entrega.

In [ ]:
# 1. Restaura o código correto da calculadora
code_fixed = '''\
function somar(a, b) {
  return a + b;
}
module.exports = { somar };
'''

with open('src/calculadora.js', 'w') as f:
    f.write(code_fixed)

print("✅ Código corrigido com sucesso!")

# Validando a esteira completa (Linter + Teste)
!npm run lint && npm test

# Compactação e entrega do artefato no ambiente do Colab
!mkdir -p ./dist_final
!cp -r src/ ./dist_final/
!tar -czvf build_entrega_aula5.tar.gz ./dist_final/

print("\n--- Artefato Final Gerado ---")
!ls -lh build_entrega_aula5.tar.gz

## ETAPA FINAL: DEPLOY AUTOMATIZADO NA PASTA LOCAL DO AMBIENTE DE EXECUÇÃO

**Desafio Extra:** Nesta etapa, você automatizará o **Step de Deploy** para enviar o pacote compilado/homologado para uma pasta de deploy criada diretamente no sistema de arquivos local do ambiente do Colab (por exemplo, `/content/deploy_local/`).

In [ ]:
import os
import time
import json

# 1. Garante que estamos na pasta correta do projeto
if os.path.exists('/content/devops_aula4ceub'):
    os.chdir('/content/devops_aula4ceub')

print(f"Diretório atual: {os.getcwd()}")

# 2. Garante a restauração do código correto da calculadora
code_fixed = '''\
function somar(a, b) {
  return a + b;
}
module.exports = { somar };
'''

with open('src/calculadora.js', 'w') as f:
    f.write(code_fixed)

# 3. Garante que as dependências do Node estão instaladas
os.system("npm install > /dev/null 2>&1")

# 4. Configurações de Deploy
NOME_ALUNO = "Danilo Pereira"  # Substitua pelo seu nome
LOCAL_DEPLOY_DIR = f"/content/deploy_local/{NOME_ALUNO}"

print("\n🚀 Iniciando Pipeline de CI/CD com Deploy Automatizado no Ambiente Local...")

# Executa os testes e captura os códigos de retorno
linter_status = os.system("npm run lint")
test_status = os.system("npm test")

if linter_status == 0 and test_status == 0:
    print("\n✅ CI Aprovado (Linter & Testes VERDES)! Iniciando etapa de Deploy na pasta local...")

    # Cria o pacote compactado caso não exista
    os.system("mkdir -p ./dist_final && cp -r src/ ./dist_final/")
    os.system("tar -czvf build_entrega_aula5.tar.gz ./dist_final/ > /dev/null 2>&1")

    # Cria a pasta local e copia os arquivos
    os.system(f"mkdir -p {LOCAL_DEPLOY_DIR}")
    os.system(f"cp -r src/* {LOCAL_DEPLOY_DIR}/")
    os.system(f"cp build_entrega_aula5.tar.gz {LOCAL_DEPLOY_DIR}/")

    # Manifesto de deploy
    manifest = {
        "aluno": NOME_ALUNO,
        "status": "DEPLOYED_SUCCESS",
        "timestamp": time.ctime(),
        "ambiente": "Execucao Local - Homologacao/Producao",
        "build_file": "build_entrega_aula5.tar.gz"
    }

    manifest_path = f"{LOCAL_DEPLOY_DIR}/manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"\n🎉 DEPLOY CONCLUÍDO COM SUCESSO!")
    print(f"📍 Local no ambiente de execução: {LOCAL_DEPLOY_DIR}")
    print("\n--- Conteúdo da Pasta de Deploy Local ---")
    os.system(f"ls -la '{LOCAL_DEPLOY_DIR}'")
else:
    print("\n❌ FALHA NO PIPELINE DE CI: O Deploy foi ABORTADO para evitar publicação de código quebrado!")

In [ ]:
import os
import shutil
import time
import json

# 1. Garante o diretório correto
if os.path.exists('/content/devops_aula4ceub'):
    os.chdir('/content/devops_aula4ceub')

print(f"Diretório atual: {os.getcwd()}")

# 2. Recria o código da calculadora em src/
os.makedirs('src', exist_ok=True)
code_calc = '''function somar(a, b) {
  return a + b;
}

module.exports = { somar };
'''
with open('src/calculadora.js', 'w') as f:
    f.write(code_calc)

# 3. Cria a estrutura na pasta __tests__ (onde o Jest realmente procura)
if os.path.exists('__tests__'):
    shutil.rmtree('__tests__')
os.makedirs('__tests__', exist_ok=True)

code_test = '''const { somar } = require('../src/calculadora');

test('Deve somar dois numeros corretamente', () => {
  expect(somar(2, 3)).toBe(5);
});
'''
with open('__tests__/calculadora.test.js', 'w') as f:
    f.write(code_test)

print("✅ Estrutura de código e arquivo __tests__/calculadora.test.js criada!")

# 4. Executa Linter e Testes
print("\n--- [VALIDAÇÃO] Linter ---")
linter_status = os.system("npm run lint")

print("\n--- [VALIDAÇÃO] Testes (Jest) ---")
test_status = os.system("npm test")

print(f"\nResultado -> Linter: {linter_status} | Testes: {test_status}")

# 5. Executa Deploy Local
NOME_ALUNO = "seu_nome_aluno"  # Substitua pelo seu nome (ex: joao_silva)
LOCAL_DEPLOY_DIR = f"/content/deploy_local/{NOME_ALUNO}"

if linter_status == 0 and test_status == 0:
    # Empacota os artefatos
    os.system("mkdir -p ./dist_final && cp -r src/ ./dist_final/")
    os.system("tar -czvf build_entrega_aula5.tar.gz ./dist_final/")

    # Copia para a pasta local
    os.system(f"mkdir -p '{LOCAL_DEPLOY_DIR}'")
    os.system(f"cp -r src/* '{LOCAL_DEPLOY_DIR}/'")
    os.system(f"cp build_entrega_aula5.tar.gz '{LOCAL_DEPLOY_DIR}/'")

    # Gera manifesto
    manifest = {
        "aluno": NOME_ALUNO,
        "status": "DEPLOYED_SUCCESS",
        "timestamp": time.ctime(),
        "ambiente": "Execucao Local - Homologacao/Producao",
        "build_file": "build_entrega_aula5.tar.gz"
    }

    with open(f"{LOCAL_DEPLOY_DIR}/manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"\n🎉 DEPLOY CONCLUÍDO COM SUCESSO!")
    print(f"📍 Pasta local de deploy criada em: {LOCAL_DEPLOY_DIR}")
    print("\n--- Arquivos na pasta de Deploy ---")
    os.system(f"ls -la '{LOCAL_DEPLOY_DIR}'")
else:
    print("\n❌ A validação ainda falhou. Verifique se o Linter/Jest exibiu erros acima.")